# WESAD — Train Teacher CNN (Session 1)

**Before running:**
1. Notebook Settings -> Accelerator -> **GPU T4 x2** (or P100).
2. Notebook Settings -> Internet -> **On** (needed for `git clone` and `pip install`).
3. Add Data -> attach your private WESAD dataset (the 15 `S{id}/S{id}.pkl` files).

This trains the 1D-CNN teacher under LOSO cross-validation (15 folds) and saves
`outputs/models/teacher_loso_S*.pt` — required before running student distillation
in the second notebook.

In [ ]:
import os, glob

# Auto-detect the attached WESAD dataset — looks for the S2/S2.pkl marker file
# under /kaggle/input so we don't need to hardcode the dataset slug.
candidates = glob.glob('/kaggle/input/*/S2/S2.pkl') + glob.glob('/kaggle/input/*/*/S2/S2.pkl')
assert candidates, (
    "WESAD dataset not found under /kaggle/input. "
    "Attach it via 'Add Data' first (must contain S2/S2.pkl ... S17/S17.pkl)."
)
data_root = os.path.dirname(os.path.dirname(candidates[0]))
print('Detected WESAD data root:', data_root)

os.environ['WESAD_DATA_DIR'] = data_root
os.environ['WESAD_OUTPUT_DIR'] = '/kaggle/working/outputs'

In [ ]:
REPO_DIR = '/kaggle/working/healthcare_wesad'
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/RiverRover-stack/healthcare_wesad.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
%cd {REPO_DIR}

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# Sanity checks — fail fast before committing to a multi-hour run.
import sys
sys.path.insert(0, 'src')

import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (!)')
assert torch.cuda.is_available(), 'GPU not enabled — check Notebook Settings -> Accelerator.'

from config import DATA_DIR, OUTPUT_DIR, ALL_SUBJECTS
print('DATA_DIR:', DATA_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)

from data import load_all_subjects
_probe = load_all_subjects(subject_ids=['S2'])
assert 'S2' in _probe, 'Failed to load S2 — check dataset path/structure.'
print('Sanity check passed: S2 loaded, ECG shape', _probe['S2'].chest_ecg.shape)

In [ ]:
# Runs the same entry point as local training — no logic duplicated here.
!python train_teacher.py

In [ ]:
# Verify all 15 folds produced a checkpoint, then zip for easy download.
import glob
ckpts = sorted(glob.glob('/kaggle/working/outputs/models/teacher_loso_S*.pt'))
print(f'{len(ckpts)} teacher checkpoints found:')
for c in ckpts:
    print(' ', c)
assert len(ckpts) == 15, f'Expected 15 checkpoints, found {len(ckpts)} — check the log above for skipped folds.'

!cd /kaggle/working && zip -rq teacher_outputs.zip outputs/models outputs/reports
print('Saved /kaggle/working/teacher_outputs.zip — download it from the notebook Output tab.')